# Code-defined workflow patterns

## Learning goals

- Implement prompt chaining, routing, parallelization, orchestrator-workers, and evaluator-optimizer
- Choose a workflow when control flow is known in advance
- Make intermediate state, limits, and failures observable
- Bound concurrency and refinement loops
- Distinguish deterministic orchestration from an agent loop


## Workflow versus agent

A **workflow** keeps the important control-flow decisions in code. The model may classify, extract, draft, or evaluate, but the application decides which steps exist and how they connect.

An **agent loop** gives the model more discretion to choose the next action at runtime. That flexibility is useful for open-ended tasks, but it adds cost, latency, and failure modes.

> Prefer the least dynamic design that solves the problem. If the path is known, encode it as a workflow.


## Mental model

```mermaid
flowchart TD
  A[Known business goal] --> B{What varies?}
  B -->|Ordered transformations| C[Prompt chain]
  B -->|One of several known paths| D[Router]
  B -->|Independent subtasks| E[Parallel workers]
  B -->|Task-specific decomposition| F[Orchestrator-workers]
  B -->|Quality needs iterative feedback| G[Evaluator-optimizer]
  C --> H[Validated result]
  D --> H
  E --> H
  F --> H
  G --> H
```

These patterns compose. An orchestrator may create parallel tasks, and each worker may use a short prompt chain. Keep each layer explicit so limits and failures remain understandable.


## 1. Prompt chaining

A chain splits a difficult transformation into ordered stages. Each stage should have a narrow contract that can be inspected before the next stage runs. Do not chain merely to look sophisticated; every extra model call adds latency and another failure point.


In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class TripRequirements:
    city: str
    days: int
    interests: tuple[str, ...]


def extract_requirements(request: str) -> TripRequirements:
    # Deterministic stand-in for a structured extraction model call.
    return TripRequirements(
        city="Jaipur",
        days=2,
        interests=("architecture", "vegetarian food"),
    )


def draft_plan(requirements: TripRequirements) -> str:
    interests = " and ".join(requirements.interests)
    return f"{requirements.days}-day {requirements.city} plan focused on {interests}."


requirements = extract_requirements("Plan two relaxed days in Jaipur...")
plan = draft_plan(requirements)
print(requirements)
print(plan)


## 2. Routing

A router chooses among a finite set of known paths. Validate the route against an allowlist and provide a safe fallback. Routing is useful when specialists need different prompts, tools, models, or service-level policies.


In [ ]:
from collections.abc import Callable


def classify_request(text: str) -> str:
    lowered = text.lower()
    if "weather" in lowered or "temperature" in lowered:
        return "weather"
    if "train" in lowered or "flight" in lowered:
        return "transport"
    return "itinerary"


ROUTES: dict[str, Callable[[str], str]] = {
    "weather": lambda text: f"Weather specialist received: {text}",
    "transport": lambda text: f"Transport specialist received: {text}",
    "itinerary": lambda text: f"Itinerary specialist received: {text}",
}


def route_request(text: str) -> str:
    route = classify_request(text)
    handler = ROUTES.get(route)
    if handler is None:
        raise ValueError(f"Unsupported route: {route}")
    return handler(text)


print(route_request("What will Jaipur weather be like?"))


## 3. Parallelization

Run independent subtasks concurrently when wall-clock latency matters. Parallel work does not reduce the total number of model calls, and unlimited concurrency can trigger rate limits or exhaust connections. A semaphore makes the limit explicit.


In [ ]:
import asyncio


async def research_topic(topic: str, semaphore: asyncio.Semaphore) -> str:
    async with semaphore:
        await asyncio.sleep(0.01)  # Simulated network or model latency.
        return f"researched:{topic}"


async def research_in_parallel(topics: list[str], concurrency: int = 2) -> list[str]:
    if concurrency < 1:
        raise ValueError("concurrency must be at least 1")
    semaphore = asyncio.Semaphore(concurrency)
    tasks = [research_topic(topic, semaphore) for topic in topics]
    return await asyncio.gather(*tasks)


parallel_results = await research_in_parallel([
    "forts",
    "vegetarian restaurants",
    "local transport",
], concurrency=2)
print(parallel_results)


## 4. Orchestrator-workers

An orchestrator creates task-specific work units, workers execute them, and a synthesizer combines results. Unlike a fixed fan-out, the number and kind of tasks can depend on the request. Validate the plan before dispatch: cap worker count, restrict task types, and deduplicate work.


In [ ]:
@dataclass(frozen=True)
class WorkItem:
    topic: str
    question: str


def plan_work(request: str, max_workers: int = 4) -> list[WorkItem]:
    proposed = [
        WorkItem("sights", "Which major sights fit a relaxed two-day plan?"),
        WorkItem("food", "Which vegetarian foods are characteristic of Jaipur?"),
        WorkItem("logistics", "How should a visitor move between the selected areas?"),
    ]
    return proposed[:max_workers]


def run_worker(item: WorkItem) -> dict[str, str]:
    return {"topic": item.topic, "finding": f"Mock finding for: {item.question}"}


def synthesize(findings: list[dict[str, str]]) -> str:
    return "\n".join(f"- {item['topic']}: {item['finding']}" for item in findings)


work_items = plan_work("Plan two relaxed days in Jaipur")
worker_results = [run_worker(item) for item in work_items]
print(synthesize(worker_results))


## 5. Evaluator-optimizer

One component drafts; another checks explicit criteria; the optimizer revises using actionable feedback. This pattern needs a hard round limit and a useful evaluator. “Make it better” is not an evaluation rubric.


In [ ]:
@dataclass(frozen=True)
class Evaluation:
    passed: bool
    feedback: tuple[str, ...]


def evaluate_plan(draft: str) -> Evaluation:
    feedback = []
    if "morning" not in draft.lower():
        feedback.append("Add a morning activity.")
    if "break" not in draft.lower():
        feedback.append("Add a rest break.")
    return Evaluation(passed=not feedback, feedback=tuple(feedback))


def revise_plan(draft: str, feedback: tuple[str, ...]) -> str:
    additions = []
    if "Add a morning activity." in feedback:
        additions.append("Morning: visit a major sight.")
    if "Add a rest break." in feedback:
        additions.append("Break: rest after lunch.")
    return "\n".join([draft, *additions])


def optimize(draft: str, max_rounds: int = 2) -> tuple[str, list[Evaluation]]:
    evaluations = []
    for _ in range(max_rounds + 1):
        evaluation = evaluate_plan(draft)
        evaluations.append(evaluation)
        if evaluation.passed:
            return draft, evaluations
        if len(evaluations) > max_rounds:
            break
        draft = revise_plan(draft, evaluation.feedback)
    return draft, evaluations


optimized, evaluation_trace = optimize("Day 1: explore Jaipur.")
print(optimized)
print("Evaluation rounds:", len(evaluation_trace))


## Pattern selection guide

| Need | Start with | Key limit |
|---|---|---|
| Ordered transformations | Prompt chain | Maximum stages and per-stage validation |
| Known specialist paths | Router | Allowed routes and fallback |
| Independent subtasks | Parallelization | Concurrency, timeout, partial-failure policy |
| Request-dependent decomposition | Orchestrator-workers | Worker count and plan validation |
| Measurable iterative quality | Evaluator-optimizer | Rounds, rubric, and no-progress detection |

When several patterns could work, choose the one with fewer model decisions and clearer failure recovery.


## Failure cases and improvements

- **Chain drift:** validate each intermediate object instead of passing vague prose forward.
- **Router collapse:** evaluate label balance and add a fallback for low-confidence cases.
- **Parallel overload:** cap concurrency and define whether one failed worker cancels or degrades the result.
- **Orchestrator explosion:** cap and deduplicate work items before dispatch.
- **Endless optimization:** set a maximum round count and stop when feedback repeats or quality no longer improves.
- **Invisible cost:** record call count, latency, token usage, retries, and failure stage.


## Exercise

1. Make one parallel worker fail and implement a documented partial-result policy.
2. Run orchestrator work items through `research_in_parallel`.
3. Add no-progress detection to the evaluator-optimizer loop.
4. Draw a workflow for a support ticket that routes, retrieves account data, drafts, and validates a reply.
5. Explain why that support flow should remain a workflow rather than a fully autonomous agent.


## Summary

Workflow patterns put control flow in code: chain ordered steps, route among known paths, parallelize independent work, orchestrate validated work plans, or iterate against a bounded rubric. Make state and limits explicit, validate every boundary, and prefer a predictable workflow whenever the task does not need open-ended action selection.
